:important: This is the notebook for my Wildfires project for SDS210. Broad structure as follows:

1. Import required packages
    

2. test pull data into project with the FIRMS API and then just use the bounding box for Australia.

3. reproject to correct CRS


4. 

In [1]:
import requests
import pandas as pd
import geopandas as gpd
import time
import folium
from folium.plugins import MarkerCluster
import numpy as np

In [2]:
# We need to access the API and to do that, will use the map key that permits access.
MAP_KEY = '54684dde74a099b139ddbbef0f621891'

# Now let's check how many results we have

url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

transaction_limit             5000
current_transactions            52
transaction_interval    10 minutes
dtype: object

In [3]:
# let's create a simple function that tells us how many transactions we have used.
# We will use this in later examples

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

tcount = get_transaction_count()
print ('Our current transaction count is %i' % tcount)

Our current transaction count is 52


In [4]:
# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
df = pd.read_csv(da_url)
display(df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-02-01,2026-05-14
1,MODIS_SP,2000-11-01,2026-01-31
2,VIIRS_NOAA20_NRT,2026-03-01,2026-05-14
3,VIIRS_NOAA20_SP,2018-04-01,2026-02-28
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-14
5,VIIRS_SNPP_NRT,2026-03-01,2026-05-14
6,VIIRS_SNPP_SP,2012-01-20,2026-02-28
7,LANDSAT_NRT,2022-06-20,2026-05-14
8,GOES_NRT,2022-08-09,2026-05-14
9,BA_MODIS,2000-11-01,2026-02-01


In [5]:
# now let's see how many transactions we use by querying this end point

start_count = get_transaction_count()
pd.read_csv(da_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

# now remember, after 10 minutes this will reset


We used 5 transactions.


In [6]:
# in this example let's look at VIIRS NOAA-20, entire world and the most recent day
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/world/1'
start_count = get_transaction_count()
df_area = pd.read_csv(area_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

df_area

We used 36 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,55.96853,160.59933,367.00,0.59,0.53,2026-05-14,56,N20,VIIRS,h,2.0NRT,310.43,50.48,D
1,64.64521,24.42407,329.33,0.39,0.36,2026-05-14,109,N20,VIIRS,n,2.0NRT,281.26,3.20,N
2,64.64832,24.42700,295.40,0.39,0.36,2026-05-14,109,N20,VIIRS,n,2.0NRT,279.35,0.68,N
3,64.65274,24.42233,308.33,0.39,0.36,2026-05-14,109,N20,VIIRS,n,2.0NRT,279.48,0.68,N
4,52.53268,39.63256,313.27,0.76,0.77,2026-05-14,111,N20,VIIRS,n,2.0NRT,260.49,2.35,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29217,51.87349,-115.14894,337.76,0.60,0.50,2026-05-14,1930,N20,VIIRS,n,2.1URT,311.49,159.84,D
29218,51.87586,-115.15975,333.11,0.60,0.50,2026-05-14,1930,N20,VIIRS,n,2.1URT,293.63,89.85,D
29219,51.87804,-115.15208,367.00,0.60,0.50,2026-05-14,1930,N20,VIIRS,h,2.1URT,354.37,185.10,D
29220,51.88054,-115.14325,331.83,0.60,0.50,2026-05-14,1930,N20,VIIRS,n,2.1URT,295.38,185.10,D


In [17]:
# We are particularly interested in the wildfires in Australia and so will select this information using a bounding box. Aus = 110 -55, 180 -10
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/110,-50,160,-12/3'
df_area = pd.read_csv(area_url)

area_url1 = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA20_NRT/110,-50,160,-12/3'
df_area1 = pd.read_csv(area_url1)

In [18]:
# Convert to GeoDataFrame (WGS84)
fires_gdf = gpd.GeoDataFrame(
    df_area, 
    geometry=gpd.points_from_xy(
        df_area["longitude"], 
        df_area["latitude"]
    ),
    crs="EPSG:4326")

fires_gdf1 = gpd.GeoDataFrame(
    df_area1, 
    geometry=gpd.points_from_xy(
        df_area1["longitude"], 
        df_area1["latitude"]
    ),
    crs="EPSG:4326")

print(f"Found {len(fires_gdf)} fire records.")
print(f"Found {len(fires_gdf1)} fire records.")

Found 1652 fire records.
Found 10021 fire records.


In [20]:
# Initialise a map centered on Australia
australia_map = folium.Map(
    location=[-28.281828, 136.145401],
    zoom_start=5,
    tiles="CyclOSM",  # A clean, light basemap
)

# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    fires_gdf,
    name = "fires",
    tooltip=folium.GeoJsonTooltip(fields=["confidence"], aliases=["Confidence (%)"]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="black",      # border color
        weight=1,           # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)

# Add raw GeoDataFrame as GeoJSON
folium.GeoJson(
    fires_gdf1,
    name = "fires1",
    tooltip=folium.GeoJsonTooltip(fields=["confidence"], aliases=["Confidence (%)"]),
    marker=folium.CircleMarker(
        radius=4,           # size in pixels
        color="blue",      # border color
        weight=1,            # border thickness
        fill=True,
        fill_color="orange",
        fill_opacity=0.7,
    ),
).add_to(australia_map)

# Add interactive layer control menu to the top right corner
folium.LayerControl().add_to(australia_map)


australia_map
